# EDA of the sqlite database

En esta libreta se hace lo siguiente:

- Explorar como acceder a la base de datos usando python y el sql necesario para hacerlo.
- Entender bien la estrucutra de las tablas, centrandonos en identificar las llaves primarias y el como se relacionan (foreign keys).

> En el futuro, tambien planeeo aqui hacer algunos analisis exploratorios de las tablas ya como dataframes de pandas.

## Libraries to use

---

First we need to import the libraries to use:

In [105]:
import sqlite3
import pandas as pd

## Creating a connection to the database and a cursor object

---

We now need to create a connection with the sqlite database and define a cursor object:

In [106]:
connection = sqlite3.connect("../INPUTS/database.sqlite")
cursor = connection.cursor()

> Note that using a cursor is not strictly necessary, for the sqlite3 library provides methods on the connection object that allow for the direct execution of queries. Behind the scenes, this methods create a temporary cursor object that is later garbage collected automatically. Still, is standard to still define cursor objects for that allows for explicit cleanup and makes the code easier to adapt to other databases like PostgreSQL.

## Three queries used to understand the *structure* of the database

---

To understand the basic structure of the database (tables, columns of tables, and how they are interconnected), we will use three sqlite queries.

#### The query "SELECT name FROM sqlite_master WHERE type='table' and name NOT LIKE 'sqlite%'"

The first thing to do to explore the database is to identify the columns it contains. We do so with the query `SELECT name FROM sqlite_master WHERE type='table' and name NOT LIKE 'sqlite%'`. This query brings tuples of whose first entry is the name of a table; filtering out names that start with the string "sqlite". 

Here we execute the code associated to said operation:

In [107]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' and name NOT LIKE 'sqlite%'")
tables = cursor.fetchall()

for table in tables:
    print(table[0])

Player_Attributes
Player
Match
League
Country
Team
Team_Attributes


Thus the database `database.sqlite` has the 7 following columns:

1. Player_Attributes
2. Player
3. Match
4. League
5. Country
6. Team
7. Team_Attributes

> Para ver que informacion es la que esta asociada a cada columna y algo de informacion de donde se obtuvo la informacion, dirigete a [esta libreta de jupyter](Libreta_resumen_bd.ipynb).

Now, that we know the tables, we will study their structure.

The `PRAGMA` command is exclusive of sqlite. According to the sqlite doccumentation, the PRAGMA statement *"...is an SQL extension specific to SQLite and used to modify the operation of the SQLite library or to query the SQLite library for internal (non-table) data.*

To explore the structure of the database (specifically its **scheme**), we'll uso two queries that rely on the *PRAGMA* command:

#### The query "PRAGMA table_info(table-name)"

What this query does is to return a list of tuples (actually a table), one tuple (i.e. one row) per each normal column in the named table. The structure of said tuples is:

1. The 'cid' column, which just represents the rank within the current result set, i.e. just the index position of the tuple within the list.
2. The 'name' of the column (string).
3. The 'type' of the column (datatype).
4. The 'notnull' column, which is just a boolean (1 if the column cannot be NULL, 0 if it can).
5. The 'dflt_value' column: The default value of the column.
6. The 'pk' column: either `0` for columns that are not part of the primary key, or the 1-based index of the column within the primary key.

Using this query will give us information about each column of a specific table within the database.

#### The query "PRAGMA foreing_key_list(table_name)"

What this query does is to return a list of tuples (actually a table), where each tuple has 8 entries (i.e. the table has 8 columns), where each describes a single column of a foreign key constraint. Said foreign key constraint and the strucure of said tuples is:

1. 'seq': The sequence number of the column within a composite key (starting from 0).
2. 'id': An integer; a unique identifier for the specific foreign key constraint within the table.
3. 'table': The name of the parent table that is referenced.
4. 'from': The name of the column in the table being queried.
5. 'to': The name of the column in the parent table.
6. 'on_update': The behavior of the foreign key constraint on update (CASCADE, SET NULL, SET DEFAULT, RESTRICT, NO ACTION).
7. 'on_delete': The behavior of the foreign key constraint on delete.
8. 'match': The match type (e.g., NONE).

Using this query will give us information about the **scheme** of the database (specifically how each table within it relates to each other).

We define two functions to apply these queries:

In [108]:
def get_columns_info(cur: sqlite3.Cursor, table_name: str) -> pd.DataFrame:
    """Returns a dataframe with the information of each column on the table 'table_name'"""

    
    columns_info = cur.execute(f"PRAGMA table_info({table_name})")
    df_columns_info = pd.DataFrame(columns_info, columns=["cid", "name", "type", "notnull", "dflt_value", "pk"]).drop(columns=["cid"])

    return df_columns_info

def get_foreing_keys(cur: sqlite3.Cursor, table_name: str) -> pd.DataFrame:
    """Returns a dataframe with the information of all foreign keys on the table 'table_name'"""

    foreign_keys = cur.execute(f"PRAGMA foreign_key_list({table_name})")
    df_foreing_keys = pd.DataFrame(foreign_keys, columns=["seq", "id", "table", "from", "to", "on_update", "on_delete", "match"]).drop(columns=["seq"])

    return df_foreing_keys

## Using the defined functions to understand the database schema

We now run the two functions for each column and do basic pandas data wrangling to understand the table schema:

### The **Player_Attributes** table


In [109]:
display(get_columns_info(cursor, "Player_Attributes"))
display(get_foreing_keys(cursor, "Player_Attributes"))

,name,type,notnull,dflt_value,pk
0,id,INTEGER,0,None,1
1,player_fifa_api_id,INTEGER,0,None,0
2,player_api_id,INTEGER,0,None,0
3,date,TEXT,0,None,0
4,overall_rating,INTEGER,0,None,0
5,potential,INTEGER,0,None,0
6,preferred_foot,TEXT,0,None,0
7,attacking_work_rate,TEXT,0,None,0
8,defensive_work_rate,TEXT,0,None,0
9,crossing,INTEGER,0,None,0


,id,table,from,to,on_update,on_delete,match
0,0,Player,player_api_id,player_api_id,NO ACTION,NO ACTION,NONE
1,0,Player,player_fifa_api_id,player_fifa_api_id,NO ACTION,NO ACTION,NONE


From the previous results, we can see that the primary key of the table `Player_Attributes` is the column `id`.

We can also see that it has two foreing key columns:

- The column `player_api_id` which is a candiate key in the table `Player`, in which the column is named the same.
- The column `player_fifa_api_id` which is a candidate key in the table `Player`, in which the column is named the same.

In a way, the `Player_Attributes` table has a child-parent relationship with the table `Player`.


### The **Player** table


In [110]:
display(get_columns_info(cursor, "Player"))
display(get_foreing_keys(cursor, "Player"))

,name,type,notnull,dflt_value,pk
0,id,INTEGER,0,None,1
1,player_api_id,INTEGER,0,None,0
2,player_name,TEXT,0,None,0
3,player_fifa_api_id,INTEGER,0,None,0
4,birthday,TEXT,0,None,0
5,height,INTEGER,0,None,0
6,weight,INTEGER,0,None,0


,id,table,from,to,on_update,on_delete,match


From the previous results we can see that the primary key of the table `Player` is `id`.

We can also see that there is no foreign key, so there is no reference to other columns of other tables within the `Player` table.

### The **Match** table


In [111]:
display(get_columns_info(cursor, "Match").tail(60))
display(get_foreing_keys(cursor, "Match"))

,name,type,notnull,dflt_value,pk
55,home_player_1,INTEGER,0,None,0
56,home_player_2,INTEGER,0,None,0
57,home_player_3,INTEGER,0,None,0
58,home_player_4,INTEGER,0,None,0
59,home_player_5,INTEGER,0,None,0
60,home_player_6,INTEGER,0,None,0
61,home_player_7,INTEGER,0,None,0
62,home_player_8,INTEGER,0,None,0
63,home_player_9,INTEGER,0,None,0
64,home_player_10,INTEGER,0,None,0


,id,table,from,to,on_update,on_delete,match
0,0,Player,away_player_11,player_api_id,NO ACTION,NO ACTION,NONE
1,0,Player,away_player_10,player_api_id,NO ACTION,NO ACTION,NONE
2,0,Player,away_player_9,player_api_id,NO ACTION,NO ACTION,NONE
3,0,Player,away_player_8,player_api_id,NO ACTION,NO ACTION,NONE
4,0,Player,away_player_7,player_api_id,NO ACTION,NO ACTION,NONE
5,0,Player,away_player_6,player_api_id,NO ACTION,NO ACTION,NONE
6,0,Player,away_player_5,player_api_id,NO ACTION,NO ACTION,NONE
7,0,Player,away_player_4,player_api_id,NO ACTION,NO ACTION,NONE
8,0,Player,away_player_3,player_api_id,NO ACTION,NO ACTION,NONE
9,0,Player,away_player_2,player_api_id,NO ACTION,NO ACTION,NONE


As we can see, again the `id` column is the primary key of the table `Match`.

We can also point out that there are plenty foreign keys, relating the `Match` table with the table `Player`, `Team`, `League` and `country`. The specific of the foreign keys are:

- The columns `away_player_{number}` and `home_player_{number}` for `number = {1,2, ... , 11}` all reference to the same candidate key `player_api_id` in the `Player` table.
- The columns `away_team_api_id` and `home_team_api_id` both reference the same candidate key `team_api_id` in the `Team` table.
- The column `league_id` references the column `id` in the `League` table.
- The column `country_id` references the column `id` in the `Country` table.

### The `League` table


In [112]:
display(get_columns_info(cursor, "League"))
display(get_foreing_keys(cursor, "League"))

,name,type,notnull,dflt_value,pk
0,id,INTEGER,0,None,1
1,country_id,INTEGER,0,None,0
2,name,TEXT,0,None,0


,id,table,from,to,on_update,on_delete,match
0,0,country,country_id,id,NO ACTION,NO ACTION,NONE


Again, the primary key of the table `League` is `id`.

The column `country_id` links the table to the table `Country`, where said key is in the `id` column.

### The **Country** table


In [113]:
display(get_columns_info(cursor, "Country"))
display(get_foreing_keys(cursor, "Country"))

,name,type,notnull,dflt_value,pk
0,id,INTEGER,0,None,1
1,name,TEXT,0,None,0


,id,table,from,to,on_update,on_delete,match


`id` is the primary key and it has no foreign keys.

### The **Team** table


In [114]:
display(get_columns_info(cursor, "Team"))
display(get_foreing_keys(cursor, "Team"))

,name,type,notnull,dflt_value,pk
0,id,INTEGER,0,None,1
1,team_api_id,INTEGER,0,None,0
2,team_fifa_api_id,INTEGER,0,None,0
3,team_long_name,TEXT,0,None,0
4,team_short_name,TEXT,0,None,0


,id,table,from,to,on_update,on_delete,match


`id` is the primary key and it has no foreign keys.

### The **Team_Attributes** table


In [115]:
display(get_columns_info(cursor, "Team_Attributes"))
display(get_foreing_keys(cursor, "Team_Attributes"))

,name,type,notnull,dflt_value,pk
0,id,INTEGER,0,None,1
1,team_fifa_api_id,INTEGER,0,None,0
2,team_api_id,INTEGER,0,None,0
3,date,TEXT,0,None,0
4,buildUpPlaySpeed,INTEGER,0,None,0
5,buildUpPlaySpeedClass,TEXT,0,None,0
6,buildUpPlayDribbling,INTEGER,0,None,0
7,buildUpPlayDribblingClass,TEXT,0,None,0
8,buildUpPlayPassing,INTEGER,0,None,0
9,buildUpPlayPassingClass,TEXT,0,None,0


,id,table,from,to,on_update,on_delete,match
0,0,Team,team_api_id,team_api_id,NO ACTION,NO ACTION,NONE
1,0,Team,team_fifa_api_id,team_fifa_api_id,NO ACTION,NO ACTION,NONE


The `id` column is the primary key.

It has two foreign keys `team_api_id` and `team_fifa_api_id`, each referring to their name-sake keys in the `Team` table.

## 

## EXTRA

Do not forget to close the connection with the database to avoid issues:

In [117]:
connection.close()